# Stage 1 — Data Ingestion
## Google Play Store Apps · Tier A (McKinsey Standard)

> **Business question.** *Which Play Store categories and listing configurations offer a defensible
> entry for a new app — and how much of an app's reach can actually be anticipated from its listing
> before launch?*
>
> **Stakeholder.** Head of Mobile Product / portfolio lead at a mid-size app studio, ahead of a
> next-build greenlight.

This notebook does four things and nothing else:

1. **Loads the raw file without coercing anything** — `dtype=str`, `keep_default_na=False`, so the
   literal string `"NaN"` in `Rating` survives to be *classified* at Stage 2 rather than silently
   becoming a float `NaN` at read time.
2. **Logs lineage** — source path, SHA-256, size, shape, pull timestamp.
3. **Validates the schema** — one row per rule, pass/fail, written to disk.
4. **Asserts the censoring structure** as a data-integrity gate (see below).

### Why the censoring check lives *here*, at ingestion

`DOCS/googleplaystore_dataset_review.md` reports `Rating` as *"1,474 missing, ~13.6%"* — a
missing-value problem. It is not. Google withholds the star rating until an app accumulates reviews,
so missingness is **structurally determined by the outcome we care about**. That is a property of the
data source, not a cleaning decision, so it is asserted at the gate: if a future refresh of this file
ever breaks the monotonicity, ingestion fails loudly rather than letting a corrected assumption pass
silently downstream.

**Raw data is immutable.** Nothing in this notebook writes to `googleplaystore.csv/`.

In [1]:
from __future__ import annotations

import hashlib
import json
import platform
import re
import sys
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd

RANDOM_STATE = 42

# Resolve the project root whether the notebook is run from notebooks/ or from the project root.
ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

RAW_CSV   = ROOT / "googleplaystore.csv" / "googleplaystore.csv"
RAW_DIR   = ROOT / "data" / "raw"
PROC_DIR  = ROOT / "data" / "processed"
for d in (RAW_DIR, PROC_DIR):
    d.mkdir(parents=True, exist_ok=True)

pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 40)

print(f"python      {sys.version.split()[0]} on {platform.system()}")
print(f"pandas      {pd.__version__}")
print(f"numpy       {np.__version__}")
print(f"project     {ROOT.name}")
print(f"raw source  {RAW_CSV.relative_to(ROOT)}")
print(f"exists      {RAW_CSV.exists()}")

python      3.14.6 on Windows
pandas      3.0.3
numpy       2.5.1
project     Google Play Store Apps
raw source  googleplaystore.csv\googleplaystore.csv
exists      True


## 1.1 Load — deliberately uncoerced

Two read options carry the whole stage:

| Option | Why |
|---|---|
| `dtype=str` | Every column arrives as text. `Reviews`, `Installs`, `Size`, `Price` are *stored* as strings with units, commas, and `$` signs — letting pandas guess produces silent partial coercion. |
| `keep_default_na=False` | `Rating` holds the **literal four-character string `"NaN"`** in 1,474 rows. With the default, pandas converts it to a float `NaN` indistinguishable from a genuinely blank field — and the distinction is the subject of this project. |

In [2]:
raw_bytes = RAW_CSV.read_bytes()
sha256 = hashlib.sha256(raw_bytes).hexdigest()

# F8 — confirm the file is clean UTF-8. Mojibake seen in some console renderings is a terminal
# artifact, not corruption; asserting it here means a future re-encode is caught rather than guessed.
try:
    raw_bytes.decode("utf-8")
    utf8_ok = True
    utf8_note = "decodes cleanly as UTF-8"
except UnicodeDecodeError as exc:
    utf8_ok = False
    utf8_note = f"UTF-8 decode failed: {exc}"

df_raw = pd.read_csv(RAW_CSV, dtype=str, keep_default_na=False, encoding="utf-8")

print(f"shape            {df_raw.shape[0]:,} rows x {df_raw.shape[1]} columns")
print(f"sha256           {sha256}")
print(f"size on disk     {len(raw_bytes):,} bytes")
print(f"encoding         {utf8_note}")
print()
print("columns:", list(df_raw.columns))
df_raw.head(3)

shape            10,841 rows x 13 columns
sha256           3e438f48161961933d26e99a8d9fc8ed79edfaa9fb34f8838e1ab4ec7a9fab91
size on disk     1,360,155 bytes
encoding         decodes cleanly as UTF-8

columns: ['App', 'Category', 'Rating', 'Reviews', 'Size', 'Installs', 'Type', 'Price', 'Content Rating', 'Genres', 'Last Updated', 'Current Ver', 'Android Ver']


,App,Category,Rating,Reviews,Size,Installs,Type,Price,Content Rating,Genres,Last Updated,Current Ver,Android Ver
0,Photo Editor & Candy Camera & Grid & ScrapBook,ART_AND_DESIGN,4.1,159,19M,"10,000+",Free,0,Everyone,Art & Design,"January 7, 2018",1.0.0,4.0.3 and up
1,Coloring book moana,ART_AND_DESIGN,3.9,967,14M,"500,000+",Free,0,Everyone,Art & Design;Pretend Play,"January 15, 2018",2.0.0,4.0.3 and up
2,"U Launcher Lite – FREE Live Cool Themes, Hide ...",ART_AND_DESIGN,4.7,87510,8.7M,"5,000,000+",Free,0,Everyone,Art & Design,"August 1, 2018",1.2.4,4.0.3 and up


## 1.2 Lineage log

Source → hash → shape → timestamp, written to `data/raw/_lineage.csv`. This is the first link in the
lineage chain the Tier A checklist requires; Stage 2 appends the transformation log and Stage 7 cites
both in the appendix.

In [3]:
lineage = pd.DataFrame([{
    "artifact":        "googleplaystore.csv",
    "source_url":      "https://www.kaggle.com/lava18/google-play-store-apps",
    "source_citation": "L. Gupta, 'Google Play Store Apps', Kaggle, Feb 2019",
    "local_path":      str(RAW_CSV.relative_to(ROOT)).replace("\\", "/"),
    "sha256":          sha256,
    "bytes":           len(raw_bytes),
    "n_rows":          int(df_raw.shape[0]),
    "n_cols":          int(df_raw.shape[1]),
    "encoding":        "utf-8",
    "read_options":    "dtype=str; keep_default_na=False",
    "pulled_utc":      datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ"),
    "immutable":       True,
}])
lineage.to_csv(RAW_DIR / "_lineage.csv", index=False)
lineage.T.rename(columns={0: "value"})

,value
artifact,googleplaystore.csv
source_url,https://www.kaggle.com/lava18/google-play-stor...
source_citation,"L. Gupta, 'Google Play Store Apps', Kaggle, Fe..."
local_path,googleplaystore.csv/googleplaystore.csv
sha256,3e438f48161961933d26e99a8d9fc8ed79edfaa9fb34f8...
bytes,1360155
n_rows,10841
n_cols,13
encoding,utf-8
read_options,dtype=str; keep_default_na=False


## 1.3 Schema validation

One row per rule, pass/fail, written to `data/raw/_schema_validation.csv`.

Two rules deserve comment because they are *count* assertions rather than *existence* assertions:

- **`Category` matches `^[A-Z_]+$`** is the corrupted-row detector. The review documents one shifted
  row (index 10472, "Life Made WI-Fi Touchscreen Photo Frame"). Asserting the count is **exactly 1**
  means a *second* corrupted row in any future refresh fails the gate instead of being absorbed into
  a `dropna()` somewhere downstream.
- **`Installs` ∈ the 21-value ladder ∪ {"Free"}** pins the bracket vocabulary. The data review states
  the field has *"only 10 discrete brackets"*; it has **21**. Since the whole install analysis is
  ordinal over this ladder, the ladder is validated rather than assumed.

In [4]:
INSTALL_LADDER = [
    "0", "0+", "1+", "5+", "10+", "50+", "100+", "500+", "1,000+", "5,000+", "10,000+",
    "50,000+", "100,000+", "500,000+", "1,000,000+", "5,000,000+", "10,000,000+",
    "50,000,000+", "100,000,000+", "500,000,000+", "1,000,000,000+",
]
EXPECTED_COLS = ["App", "Category", "Rating", "Reviews", "Size", "Installs", "Type", "Price",
                 "Content Rating", "Genres", "Last Updated", "Current Ver", "Android Ver"]

rules = []
def check(rule, expected, observed, passed, note=""):
    rules.append({"rule": rule, "expected": expected, "observed": observed,
                  "status": "PASS" if passed else "FAIL", "note": note})

# --- structure -------------------------------------------------------------------------------
check("shape is 10,841 x 13", "10841 x 13", f"{df_raw.shape[0]} x {df_raw.shape[1]}",
      df_raw.shape == (10841, 13))
check("column names and order match the data dictionary", "13 named columns",
      "match" if list(df_raw.columns) == EXPECTED_COLS else "mismatch",
      list(df_raw.columns) == EXPECTED_COLS)
check("file decodes as UTF-8", "clean decode", utf8_note, utf8_ok, "F8")

# --- the corrupted-row detector -------------------------------------------------------------
bad_cat = ~df_raw["Category"].str.fullmatch(r"[A-Z_]+")
check("Category matches ^[A-Z_]+$ for all but exactly 1 row", "1 violation",
      f"{int(bad_cat.sum())} violation(s)", int(bad_cat.sum()) == 1,
      "the documented left-shifted scrape artifact")

# --- vocabulary pins --------------------------------------------------------------------------
inst_unknown = set(df_raw["Installs"].unique()) - set(INSTALL_LADDER) - {"Free"}
check("Installs vocabulary is the 21-bracket ladder (+ the corrupt row's 'Free')",
      "no unknown values", f"{len(inst_unknown)} unknown: {sorted(inst_unknown) or 'none'}",
      len(inst_unknown) == 0, "review claims 10 brackets; there are 21")
check("Installs ladder is fully populated", "21 brackets present",
      f"{df_raw['Installs'].isin(INSTALL_LADDER).pipe(lambda s: df_raw.loc[s,'Installs'].nunique())} present",
      df_raw.loc[df_raw["Installs"].isin(INSTALL_LADDER), "Installs"].nunique() == 21)

type_unknown = set(df_raw["Type"].unique()) - {"Free", "Paid", "NaN", "0"}
check("Type vocabulary is {Free, Paid, NaN, 0}", "no unknown values",
      f"{len(type_unknown)} unknown: {sorted(type_unknown) or 'none'}", len(type_unknown) == 0,
      "'0' is the corrupt row; 'NaN' is one genuine blank")

bad_price = ~df_raw["Price"].str.fullmatch(r"\$?[\d.]+")
check("Price matches ^\\$?[\\d.]+$ for all but the corrupt row", "1 violation",
      f"{int(bad_price.sum())} violation(s)", int(bad_price.sum()) == 1)

# --- Rating domain -----------------------------------------------------------------------------
rating_num = pd.to_numeric(df_raw["Rating"], errors="coerce")
literal_nan = int((df_raw["Rating"] == "NaN").sum())
out_of_range = int(((rating_num < 1.0) | (rating_num > 5.0)).sum())
check("Rating is numeric in [1.0, 5.0] or the literal string 'NaN'", "1 out-of-range (corrupt row)",
      f"{out_of_range} out-of-range; {literal_nan:,} literal 'NaN'", out_of_range == 1,
      "the 19.0 belongs to the same row Category caught")

# --- dates -------------------------------------------------------------------------------------
lu = pd.to_datetime(df_raw["Last Updated"], format="%B %d, %Y", errors="coerce")
# The corrupt row's left shift puts a version string ("1.0.19") in Last Updated, so exactly one
# failure is expected - and it must be the *same* row Category caught, not a separate date problem.
lu_fail_is_corrupt = set(df_raw.index[lu.isna()]) == set(df_raw.index[bad_cat])
check("Last Updated parses under '%B %d, %Y' for all but the corrupt row", "1 failure, same row",
      f"{int(lu.isna().sum())} failure(s); same row as Category violation: {lu_fail_is_corrupt}",
      int(lu.isna().sum()) == 1 and lu_fail_is_corrupt,
      "the shift puts a version string in the date field")
check("Last Updated max is 2018-08-08 (the scrape vintage)", "2018-08-08",
      str(lu.max().date()), lu.max().date().isoformat() == "2018-08-08",
      "F7 - review header says 'early 2019'; that is the Kaggle publication date")

schema = pd.DataFrame(rules)
schema.to_csv(RAW_DIR / "_schema_validation.csv", index=False)

n_fail = int((schema["status"] == "FAIL").sum())
print(f"schema rules: {len(schema)} checked, {len(schema) - n_fail} passed, {n_fail} failed\n")
schema

schema rules: 11 checked, 11 passed, 0 failed



,rule,expected,observed,status,note
0,"shape is 10,841 x 13",10841 x 13,10841 x 13,PASS,
1,column names and order match the data dictionary,13 named columns,match,PASS,
2,file decodes as UTF-8,clean decode,decodes cleanly as UTF-8,PASS,F8
3,Category matches ^[A-Z_]+$ for all but exactly...,1 violation,1 violation(s),PASS,the documented left-shifted scrape artifact
4,Installs vocabulary is the 21-bracket ladder (...,no unknown values,0 unknown: none,PASS,review claims 10 brackets; there are 21
5,Installs ladder is fully populated,21 brackets present,21 present,PASS,
6,"Type vocabulary is {Free, Paid, NaN, 0}",no unknown values,0 unknown: none,PASS,'0' is the corrupt row; 'NaN' is one genuine b...
7,Price matches ^\$?[\d.]+$ for all but the corr...,1 violation,1 violation(s),PASS,
8,"Rating is numeric in [1.0, 5.0] or the literal...",1 out-of-range (corrupt row),"1 out-of-range; 1,474 literal 'NaN'",PASS,the 19.0 belongs to the same row Category caught
9,"Last Updated parses under '%B %d, %Y' for all ...","1 failure, same row",1 failure(s); same row as Category violation: ...,PASS,the shift puts a version string in the date field


In [5]:
assert n_fail == 0, f"schema validation failed on {n_fail} rule(s) - inspect _schema_validation.csv"
print("Schema gate PASSED.")

Schema gate PASSED.


## 1.4 The corrupted row, on the record

Exactly one row fails the `Category` rule. Its columns are shifted **left by one** from `Category`
onward — the classic single-field scrape dropout. It is displayed here (not repaired, not dropped)
so that the Stage 2 repair-versus-drop decision is made against the visible evidence.

It also accounts for the two stray vocabulary values the data review flags separately: `Type == "0"`
and `Installs == "Free"`. The review guesses these are *"likely the same corrupted row"* — they are.

In [6]:
corrupt = df_raw[bad_cat]
print(f"corrupted rows: {len(corrupt)} (positional index {list(corrupt.index)})\n")
display(corrupt.T.rename(columns=lambda c: f"row {c}"))

print("The two stray vocabulary values the review flags separately:")
print(f"  rows with Type    == '0'    : {int((df_raw['Type'] == '0').sum())}  "
      f"-> same row: {bool((df_raw['Type'] == '0').equals(bad_cat & (df_raw['Type'] == '0')))}")
print(f"  rows with Installs== 'Free' : {int((df_raw['Installs'] == 'Free').sum())}")
print(f"  both are the corrupted row  : "
      f"{set(df_raw.index[df_raw['Type'] == '0']) == set(corrupt.index) == set(df_raw.index[df_raw['Installs'] == 'Free'])}")

corrupted rows: 1 (positional index [10472])



,row 10472
App,Life Made WI-Fi Touchscreen Photo Frame
Category,1.9
Rating,19
Reviews,3.0M
Size,"1,000+"
Installs,Free
Type,0
Price,Everyone
Content Rating,
Genres,"February 11, 2018"


The two stray vocabulary values the review flags separately:
  rows with Type    == '0'    : 1  -> same row: True
  rows with Installs== 'Free' : 1
  both are the corrupted row  : True


## 1.5 The censoring gate — `Rating` is not missing at random

This is the integrity assertion that shapes the rest of the project.

If `Rating` were missing at random, the missing rate would be flat across review counts and install
brackets. It is not: it is **100% for apps with zero reviews** and falls **monotonically to 0%** for
apps above ten thousand reviews. Google does not publish a star rating below a review threshold.

The gate asserts three things:

1. Every zero-review app is missing its rating (**100%, exactly**).
2. The missing rate is **monotone non-increasing** across the 21-bracket install ladder.
3. A **Cochran–Armitage trend test** over the ladder is significant — so the monotonicity is a trend,
   not an artifact of small cells at the ends.

Failing any of these means the data source changed its behaviour, and every rating statistic
downstream would need re-deriving.

In [7]:
gate = df_raw[~bad_cat].copy()                       # exclude the corrupt row from the mechanism test
gate["rating_missing"] = (gate["Rating"] == "NaN")
gate["reviews_int"]    = pd.to_numeric(gate["Reviews"], errors="coerce")

# --- (1) zero-review apps ----------------------------------------------------------------------
zero_rev = gate[gate["reviews_int"] == 0]
zero_rev_missing = float(zero_rev["rating_missing"].mean())

# --- review-bucket staircase -------------------------------------------------------------------
buckets = pd.cut(gate["reviews_int"], [-1, 0, 10, 100, 1_000, 10_000, np.inf],
                 labels=["0", "1-10", "11-100", "101-1k", "1k-10k", ">10k"])
by_reviews = (gate.groupby(buckets, observed=True)["rating_missing"]
                  .agg(apps="size", missing="sum", missing_pct="mean"))
by_reviews["missing_pct"] = (by_reviews["missing_pct"] * 100).round(1)

# --- install-ladder staircase ------------------------------------------------------------------
by_installs = (gate.groupby("Installs", observed=True)["rating_missing"]
                   .agg(apps="size", missing="sum", missing_pct="mean"))
by_installs["missing_pct"] = (by_installs["missing_pct"] * 100).round(1)
by_installs = by_installs.reindex([b for b in INSTALL_LADDER if b in by_installs.index])

print(f"overall Rating missing : {gate['rating_missing'].mean()*100:.1f}%  "
      f"({int(gate['rating_missing'].sum()):,} of {len(gate):,})")
print(f"zero-review apps       : {len(zero_rev):,}, of which {zero_rev_missing*100:.1f}% have no rating\n")
print("--- missing rate by review bucket ---")
print(by_reviews.to_string())
print("\n--- missing rate by install bracket ---")
print(by_installs.to_string())

overall Rating missing : 13.6%  (1,474 of 10,840)
zero-review apps       : 596, of which 100.0% have no rating

--- missing rate by review bucket ---
             apps  missing  missing_pct
reviews_int                            
0             596      596        100.0
1-10         1296      654         50.5
11-100       1485      191         12.9
101-1k       1562       28          1.8
1k-10k       1621        5          0.3
>10k         4280        0          0.0

--- missing rate by install bracket ---
                apps  missing  missing_pct
Installs                                  
0                  1        1        100.0
0+                14       14        100.0
1+                67       64         95.5
5+                82       73         89.0
10+              386      317         82.1
50+              205      149         72.7
100+             719      410         57.0
500+             330      129         39.1
1,000+           907      194         21.4
5,000+          

In [8]:
from scipy import stats

# --- (3) Cochran-Armitage trend test across the ordered install ladder --------------------------
# Implemented directly: the CA statistic is the score test for a linear trend in proportions.
tbl   = by_installs.dropna()
score = np.arange(len(tbl), dtype=float)          # ordinal position on the ladder
n_i   = tbl["apps"].to_numpy(dtype=float)
x_i   = tbl["missing"].to_numpy(dtype=float)
N, X  = n_i.sum(), x_i.sum()
p_bar = X / N
s_bar = (n_i * score).sum() / N
num   = (score * (x_i - n_i * p_bar)).sum()
den   = np.sqrt(p_bar * (1 - p_bar) * (n_i * (score - s_bar) ** 2).sum())
z_ca  = num / den
p_ca  = 2 * stats.norm.sf(abs(z_ca))

# --- (2) monotonicity across the ladder ---------------------------------------------------------
rates = tbl["missing_pct"].to_numpy()
monotone = bool(np.all(np.diff(rates) <= 1e-9))
violations = [(tbl.index[i], rates[i], rates[i + 1]) for i in range(len(rates) - 1)
              if rates[i + 1] > rates[i] + 1e-9]

print(f"(1) zero-review apps missing a rating : {zero_rev_missing*100:.1f}%")
print(f"(2) monotone non-increasing on ladder : {monotone}"
      + (f"   violations: {violations}" if violations else ""))
print(f"(3) Cochran-Armitage trend            : z = {z_ca:.1f}, p = {p_ca:.3g}")

assert zero_rev_missing == 1.0, "a zero-review app now carries a rating - the gating rule changed"
assert monotone, f"missing-rate monotonicity broken on the install ladder: {violations}"
assert p_ca < 1e-10, "the censoring trend is no longer significant"
print("\nCensoring gate PASSED - Rating is structurally censored, not missing at random.")

(1) zero-review apps missing a rating : 100.0%
(2) monotone non-increasing on ladder : True
(3) Cochran-Armitage trend            : z = -61.2, p = 0

Censoring gate PASSED - Rating is structurally censored, not missing at random.


**So What.** Every published "average rating by category" figure computed from this file — and there
are thousands — silently conditions on apps that survived long enough to accumulate reviews. The
1,474 apps without a rating are not a data-quality nuisance to be dropped; they are **the long tail
of the catalog**, and Stage 3 quantifies exactly what dropping them does to the sample.

**Implication.** `Rating` cannot be imputed (imputation would fabricate the tail), and
`Rating_Observed` becomes a first-class feature at Stage 2 rather than a filter.

## 1.6 Governance & data classification

| Item | Assessment |
|---|---|
| **PII** | **None.** The file describes app listings — product metadata only. No user, developer, or reviewer identity; no demographic, location, or behavioural attribute. |
| **Consent / purpose limitation** | Public store-catalog metadata, published by Kaggle under the source's terms for analysis. Purpose here (market-structure analysis) is consistent with that. |
| **Identifiability** | App names are **real, identifiable products**. Findings are therefore reported in aggregate and no app is named in a negative light — with one stated exception (§ below). |
| **Fairness scope** | Because the data contains **no attribute about people**, a protected-subgroup audit is *impossible*. That impossibility is carried forward as a main-narrative limitation, not an omission. |
| **Retention** | Raw file stays local and is never committed. The shipped artifacts are notebooks, the HTML report, and the README. |

**The one naming exception.** The ~20 novelty listings priced \$79.99–\$400 ("I Am Rich" and kin) are
named in the report. They are a documented, widely-known catalog phenomenon, and naming them is what
makes the Stage 2 decision to *flag rather than delete* them auditable.

In [9]:
governance = pd.DataFrame([
    {"item": "PII present",             "assessment": "No",
     "detail": "Product metadata only - no user, developer, or reviewer identity"},
    {"item": "Sensitive attributes",    "assessment": "No",
     "detail": "No demographic, location, health, or financial attribute about any person"},
    {"item": "Identifiable entities",   "assessment": "Yes - app titles",
     "detail": "Real products; findings reported in aggregate, no app named negatively "
               "except the documented novelty-price listings"},
    {"item": "Fairness audit feasible", "assessment": "No - by construction",
     "detail": "No attribute about people exists; impossibility promoted to a main-narrative "
               "limitation. Runnable proxies: model error rates by category, Content Rating mix"},
    {"item": "Purpose limitation",      "assessment": "Consistent",
     "detail": "Public catalog metadata used for market-structure analysis"},
    {"item": "Retention",               "assessment": "Local only",
     "detail": "Raw CSV never committed; ship set is notebooks + HTML report + README"},
    {"item": "Survivorship",            "assessment": "Material limitation",
     "detail": "Only apps live at scrape time are present; delisted and failed apps are absent"},
])
governance.to_csv(RAW_DIR / "_governance.csv", index=False)
governance

,item,assessment,detail
0,PII present,No,"Product metadata only - no user, developer, or..."
1,Sensitive attributes,No,"No demographic, location, health, or financial..."
2,Identifiable entities,Yes - app titles,"Real products; findings reported in aggregate,..."
3,Fairness audit feasible,No - by construction,No attribute about people exists; impossibilit...
4,Purpose limitation,Consistent,Public catalog metadata used for market-struct...
5,Retention,Local only,Raw CSV never committed; ship set is notebooks...
6,Survivorship,Material limitation,Only apps live at scrape time are present; del...


## 1.7 Hand-off to Stage 2

The raw frame is **not** written to `data/processed/` — Stage 2 re-reads the immutable source with
the same options. What ships forward from this notebook is the three audit files plus a small
constants block that Stage 2 imports by re-declaration (no local module, per the project's
fresh-clone rule).

In [10]:
handoff = {
    "sha256": sha256,
    "n_rows_raw": int(df_raw.shape[0]),
    "n_cols_raw": int(df_raw.shape[1]),
    "corrupt_row_index": [int(i) for i in corrupt.index],
    "install_ladder": INSTALL_LADDER,
    "scrape_reference_date": "2018-08-08",
    "rating_literal_nan": literal_nan,
    "rating_missing_pct": round(float(gate["rating_missing"].mean()) * 100, 2),
    "zero_review_apps": int(len(zero_rev)),
    "cochran_armitage_z": round(float(z_ca), 2),
    "schema_rules_passed": int(len(schema) - n_fail),
    "random_state": RANDOM_STATE,
}
(RAW_DIR / "_ingestion_handoff.json").write_text(json.dumps(handoff, indent=2), encoding="utf-8")

print("written to data/raw/:")
for f in sorted(RAW_DIR.glob("_*")):
    print(f"  {f.name:28s} {f.stat().st_size:>7,} bytes")
print()
print(json.dumps(handoff, indent=2)[:700] + " ...")

written to data/raw/:
  _governance.csv                  866 bytes
  _ingestion_handoff.json          750 bytes
  _lineage.csv                     434 bytes
  _schema_validation.csv         1,349 bytes

{
  "sha256": "3e438f48161961933d26e99a8d9fc8ed79edfaa9fb34f8838e1ab4ec7a9fab91",
  "n_rows_raw": 10841,
  "n_cols_raw": 13,
  "corrupt_row_index": [
    10472
  ],
  "install_ladder": [
    "0",
    "0+",
    "1+",
    "5+",
    "10+",
    "50+",
    "100+",
    "500+",
    "1,000+",
    "5,000+",
    "10,000+",
    "50,000+",
    "100,000+",
    "500,000+",
    "1,000,000+",
    "5,000,000+",
    "10,000,000+",
    "50,000,000+",
    "100,000,000+",
    "500,000,000+",
    "1,000,000,000+"
  ],
  "scrape_reference_date": "2018-08-08",
  "rating_literal_nan": 1474,
  "rating_missing_pct": 13.6,
  "zero_review_apps": 596,
  "cochran_armitage_z": -61.21,
  "schema_rules_passed": 11,
  "random ...


## Stage 1 — Gate Checklist

- [x] **Raw data saved to `data/raw/` (untouched copy)** — the source lives at
      `googleplaystore.csv/` and is read-only; SHA-256 `3e438f48…` recorded rather than copied, since
      duplicating a 1.36 MB immutable file adds nothing but drift risk.
- [x] **Source, timestamp, and row/column counts logged** — `data/raw/_lineage.csv`
      (10,841 × 13, UTF-8, read options recorded).
- [x] **Schema matches expectations (or discrepancies documented)** — 11 rules, all PASS,
      `data/raw/_schema_validation.csv`. Two review claims corrected on the record: the install
      vocabulary is **21 brackets, not 10**, and the vintage is **August 2018, not early 2019**.
- [x] **PII / data governance review completed** — `data/raw/_governance.csv`. No PII; the
      *impossibility* of a subgroup fairness audit is logged as a limitation to carry forward.
- [x] **Data-integrity gate: censoring asserted** — zero-review apps are 100% unrated, the missing
      rate is monotone across all 21 install brackets, and the Cochran–Armitage trend is significant.

**Next:** `02_cleaning.ipynb` — the corrupted-row repair-versus-drop decision, the three missingness
mechanisms, the derived-column set, and the dual-grain duplicate treatment.